In [2]:
#OPEN DATASETS CODE

In [3]:
import pandas as pd
from pathlib import Path
import numpy as np
from typing import Optional, Union, List, Tuple


class DatasetLoader:
    def __init__(self, file_path: Union[str, Path]):
        self.file_path = Path(file_path)
        self.delimiter: Optional[str] = None
        self.has_header: Optional[bool] = None
        self.encoding: Optional[str] = None
        self.dataframe: Optional[pd.DataFrame] = None

        # All samples
        self.data_samples_X: Optional[np.ndarray] = None
        self.data_samples_Y: Optional[np.ndarray] = None

        # Complete vs. missing subsets
        self.complete_X: Optional[np.ndarray] = None
        self.complete_Y: Optional[np.ndarray] = None
        self.missing_X: Optional[np.ndarray] = None
        self.missing_Y: Optional[np.ndarray] = None

    def _detect_delimiter_and_header(
        self, encoding: Optional[str] = None, sample_size: int = 5
    ) -> Optional[str]:

        often_used_delim = [',', ';', '\t', ' ', '|']
        delimiter_counts = {delim: 0 for delim in often_used_delim}

        encodings = ['utf-8', 'latin-1', 'iso-8859-1', 'cp1252']
        if encoding is not None:
            encodings = [encoding]

        for enc in encodings:
            try:
                with open(self.file_path, 'r', encoding=enc) as f:
                    self.encoding = enc
                    for i, line in enumerate(f):
                        if i >= sample_size:
                            break
                        line = line.strip()
                        if not line:
                            continue
                        for delim in often_used_delim:
                            if delim in line:
                                fields = line.split(delim)
                                if len(fields) > 1:
                                    delimiter_counts[delim] += 1
                break  # successfully opened
            except UnicodeDecodeError:
                continue

        if max(delimiter_counts.values()) > 0:
            self.delimiter = max(delimiter_counts, key=delimiter_counts.get)

        if self.delimiter:
            self._detect_header()

        return self.delimiter

    def _detect_header(self, sample_size: int = 5) -> None:
        try:
            with open(self.file_path, 'r', encoding=self.encoding) as f:
                lines = [f.readline().strip() for _ in range(sample_size)]
            lines = [line for line in lines if line]  # skip empty lines
            if len(lines) < 2:
                self.has_header = False
                return

            rows = [line.split(self.delimiter) for line in lines]
            first_row = rows[0]
            other_rows = rows[1:]

            #check 1: different column counts
            len_first = len(first_row)
            len_others = [len(r) for r in other_rows]
            if len_first != max(len_others, default=len_first):
                self.has_header = True
                return

            #check 2: typical header strings (non‑numeric)
            def is_likely_header_value(val: str) -> bool:
                val = val.strip(' "\'')
                if val == '':
                    return False
                try:
                    float(val)
                    return False
                except ValueError:
                    pass
                return any(c.isalpha() for c in val)

            first_row_header_score = sum(is_likely_header_value(v) for v in first_row)
            if first_row_header_score >= len(first_row) / 2:
                self.has_header = True
                return

            #check 3: data type signature
            def row_type_signature(row):
                return [type(v).__name__ for v in row]

            first_sig = row_type_signature(first_row)
            other_sigs = [row_type_signature(r) for r in other_rows]
            if first_sig != (other_sigs[0] if other_sigs else first_sig):
                self.has_header = True
                return

            self.has_header = False

        except Exception:
            self.has_header = False

    @staticmethod
    def convert_data_samples(samples: np.ndarray) -> np.ndarray:
        if samples is None:
            print("data_samples is None")
            return None

        missing_values = ['?', 'NA', 'N/A', 'null', 'NULL', 'None', '', ' ']
        converted_samples = samples.copy().astype(object)

        for i in range(samples.shape[1]):
            try:
                col = samples[:, i]
                # Replace missing placeholders with np.nan
                col_clean = np.where(np.isin(col, missing_values), np.nan, col)
                # Try converting to numeric
                converted = pd.to_numeric(col_clean, errors='coerce')
                numeric_ratio = (~pd.isna(converted)).sum() / len(converted)

                if numeric_ratio >= 0.5:
                    converted_samples[:, i] = converted
                else:
                    # Keep as string, empty string for missing
                    converted_samples[:, i] = np.where(
                        np.isin(col, missing_values), '', col.astype(str)
                    )
            except (ValueError, TypeError, IndexError) as e:
                print(f"Failed to process column {i}: {e}")
                continue

        return converted_samples

    def _extract_target_labels(
        self, data: np.ndarray
    ) -> Tuple[np.ndarray, Optional[np.ndarray]]:
        if data.shape[1] < 2:
            return data, None

        last_col = data[:, -1]
        try:
            pd.to_numeric(last_col, errors='raise')
            is_numeric = True
        except (ValueError, TypeError):
            is_numeric = False

        unique_vals = np.unique(last_col)
        is_categorical = len(unique_vals) <= 10

        if not is_numeric or is_categorical:
            X = data[:, :-1]
            Y = last_col
            print(f"Detected target column (last column): {len(unique_vals)} unique values")
            return X, Y
        else:
            print("No obvious target column detected. All columns treated as features.")
            return data, None

    def _split_missing_samples(self) -> None:
        if self.data_samples_X is None:
            return

        X = self.data_samples_X
        Y = self.data_samples_Y

        missing_mask = np.zeros(X.shape[0], dtype=bool)

        for col_idx in range(X.shape[1]):
            col = X[:, col_idx]
            if col.dtype.kind in 'fiu':          
                missing_mask |= pd.isna(col)
            else:                                 
                missing_mask |= (col == '')

        self.complete_X = X[~missing_mask]
        self.missing_X = X[missing_mask]

        if Y is not None:
            self.complete_Y = Y[~missing_mask]
            self.missing_Y = Y[missing_mask]
        else:
            self.complete_Y = None
            self.missing_Y = None

        print(f"Complete samples: {self.complete_X.shape[0]}")
        print(f"Samples with missing values: {self.missing_X.shape[0]}")

    def load(self, encoding: Optional[str] = None) -> None:
        if encoding:
            self._detect_delimiter_and_header(encoding)
        else:
            self._detect_delimiter_and_header()

        if not self.delimiter:
            print("Could not detect delimiter")
            return

        try:
            df = pd.read_csv(self.file_path,delimiter=self.delimiter,
                header=0 if self.has_header else None,
                encoding=self.encoding,engine='python',on_bad_lines='skip')
            
            self.dataframe = df

            # Split into X and Y
            data_array = df.to_numpy()
            X_raw, Y_raw = self._extract_target_labels(data_array)

            # Convert feature columns
            self.data_samples_X = self.convert_data_samples(X_raw)
            self.data_samples_Y = Y_raw

            # Separate complete and missing rows
            self._split_missing_samples()

            print("Dataframe loaded successfully.")
            print(f"Features shape (all): {self.data_samples_X.shape}")
            if self.data_samples_Y is not None:
                print(f"Target shape (all): {len(self.data_samples_Y)}")

        except Exception as e:
            print(f"Error loading file: {e}")

    def get_x_all(self) -> Optional[np.ndarray]:
        if self.data_samples_X is None:
            print("Missing.")
            return None
        return self.data_samples_X

    def get_x_complete(self) -> Optional[np.ndarray]:
        if self.complete_X is None:
            print("Missing.")
            return None
        return self.complete_X

    def get_x_missing(self) -> Optional[np.ndarray]:
        if self.missing_X is None:
            print("Missing.")
            return None
        return self.missing_X

    def get_y_all(self) -> Optional[np.ndarray]:
        if self.data_samples_Y is None:
            print("Missing.")
            return None
        return self.data_samples_Y

    def get_y_complete(self) -> Optional[np.ndarray]:
        if self.complete_Y is None:
            print("Missing.")
            return None
        return self.complete_Y

    def get_y_missing(self) -> Optional[np.ndarray]:
        if self.missing_Y is None:
            print("Missing.")
            return None
        return self.missing_Y


def load_dataloader_by_name(dataset_name: str, data_dir: str = 'datasets', **kwargs) -> DatasetLoader:
    base_path = Path(data_dir)
    possible_paths = [
        base_path / dataset_name / f"{dataset_name}.csv",
        base_path / dataset_name / f"{dataset_name}.data"
    ]

    for path in possible_paths:
        if path.exists():
            loader = DatasetLoader(path)
            loader.load(**kwargs)
            return loader

    raise FileNotFoundError(
        f"Dataset '{dataset_name}' not found in '{data_dir}'. "
        f"Tried: {[str(p) for p in possible_paths]}"
    )

In [4]:
dataset_names = ["bank","adult","breast_cancer","heart_disease","iris", "wine_quality", "mushroom"]
for data_name in dataset_names:
    dataset = load_dataloader_by_name(data_name)
    all_x = dataset.get_x_all()
    #print(all_x)
    complete_x = dataset.get_x_complete()
    #print(complete_x)
    missing_x = dataset.get_x_missing()
    #print(missing_x)
    all_y = dataset.get_y_all()
    #print(all_y)
    complete_y = dataset.get_y_complete()
    #print(complete_y)
    missing_y = dataset.get_y_missing()
    #print(missing_y)
values_X = dataset.get_x_complete().T
values_Y = dataset.get_y_complete().T
print(values_X)

Detected target column (last column): 2 unique values
Complete samples: 4521
Samples with missing values: 0
Dataframe loaded successfully.
Features shape (all): (4521, 16)
Target shape (all): 4521
Detected target column (last column): 2 unique values
Complete samples: 32560
Samples with missing values: 0
Dataframe loaded successfully.
Features shape (all): (32560, 14)
Target shape (all): 32560
No obvious target column detected. All columns treated as features.
Complete samples: 569
Samples with missing values: 0
Dataframe loaded successfully.
Features shape (all): (569, 32)
Missing.
Missing.
Missing.
Detected target column (last column): 5 unique values
Complete samples: 303
Samples with missing values: 0
Dataframe loaded successfully.
Features shape (all): (303, 13)
Target shape (all): 303
Detected target column (last column): 3 unique values
Complete samples: 150
Samples with missing values: 0
Dataframe loaded successfully.
Features shape (all): (150, 4)
Target shape (all): 150
Detec

In [5]:
#ClUSTERING CODE


In [6]:
from typing import List, Optional, Any, Callable, Union


class Node:
    __slots__ = ('points', 'min_val', 'max_val', 'left', 'right', 'depth')

    def __init__(self, points: List[Any], depth: int = 0):
        self.points = sorted(points)  # Keep sorted for consistent splitting
        self.min_val = self.points[0] if self.points else None
        self.max_val = self.points[-1] if self.points else None
        self.left = None
        self.right = None
        self.depth = depth


class DivisiveCluster:
    def __init__(self,max_depth: Optional[int] = None,min_cluster_size: int = 1,distance_func: Optional[Callable[[Any, Any], float]] = None):
        self.max_depth = max_depth
        self.min_cluster_size = min_cluster_size
        self.distance_func = distance_func
        self._root = None
        self._original_data = None

    def _get_gap(self, a: Any, b: Any) -> float:
        """Compute gap between two elements using the provided or default distance function."""
        if self.distance_func is not None:
            return self.distance_func(a, b)

        # Default numeric gap
        if isinstance(a, (int, float)) and isinstance(b, (int, float)):
            return float(b - a)

        # Default string gap: treat as 1.0 if strings differ, 0.0 otherwise.
        # This clusters identical strings together and separates different ones.
        if isinstance(a, str) and isinstance(b, str):
            return 1.0 if a != b else 0.0

        raise TypeError(
            f"Cannot compute default gap between {type(a).__name__} and {type(b).__name__}. "
            "Provide a custom distance_func."
        )

    def fit(self, data: List[Any]) -> None:
        if data is None or len(data) == 0:
            self._original_data = []
            self._root = None
            return

        self._original_data = data.copy()
        sorted_points = sorted(data)
        self._root = Node(sorted_points, depth=0)
        self.split_recursive(self._root)

    def split_recursive(self, node: Node) -> None:
        # Stop if max_depth reached
        if self.max_depth is not None and node.depth >= self.max_depth:
            return

        # Stop if cluster is too small to split (need at least 2 * min_cluster_size)
        if len(node.points) < 2 * self.min_cluster_size:
            return

        pts = node.points
        max_gap = -1.0
        split_idx = -1

        for i in range(len(pts) - 1):
            left_size = i + 1
            right_size = len(pts) - (i + 1)

            if left_size >= self.min_cluster_size and right_size >= self.min_cluster_size:
                gap = self._get_gap(pts[i], pts[i + 1])
                if gap > max_gap:
                    max_gap = gap
                    split_idx = i

        # No valid split found
        if split_idx == -1 or max_gap <= 0:
            return

        # Create left and right children
        left_points = pts[: split_idx + 1]
        right_points = pts[split_idx + 1 :]

        node.left = Node(left_points, node.depth + 1)
        node.right = Node(right_points, node.depth + 1)

        # Recurse only if we haven't reached max_depth
        if self.max_depth is None or node.depth + 1 < self.max_depth:
            self.split_recursive(node.left)
            self.split_recursive(node.right)

    def get_clusters(self) -> List[List[Any]]:
        if self._root is None:
            return []
        leaves = []
        self.collect_leaves(self._root, leaves)
        return [leaf.points.copy() for leaf in leaves]

    def collect_leaves(self, node: Node, leaves: List[Node]) -> None:
        if node is None:
            return
        if node.left is None and node.right is None:
            leaves.append(node)
        else:
            if node.left:
                self.collect_leaves(node.left, leaves)
            if node.right:
                self.collect_leaves(node.right, leaves)

    def get_clusters_at_depth(self, depth: int) -> List[List[Any]]:
        if self._root is None:
            return []
        nodes_at_depth = []
        self.collect_at_depth(self._root, depth, nodes_at_depth)
        return [node.points.copy() for node in nodes_at_depth]

    def collect_at_depth(self, node: Node, target_depth: int, result: List[Node]) -> None:
        if node is None:
            return
        if node.depth == target_depth:
            result.append(node)
            return
        self.collect_at_depth(node.left, target_depth, result)
        self.collect_at_depth(node.right, target_depth, result)

    def print_tree(self, node: Node = None, level: int = 0) -> None:
        """Print the clustering hierarchy."""
        if node is None:
            node = self._root
        if node is None:
            print("Empty tree")
            return
        indent = "  " * level
        # Format min/max as strings for display; numbers get limited precision
        min_str = f"{node.min_val:.2f}" if isinstance(node.min_val, float) else str(node.min_val)
        max_str = f"{node.max_val:.2f}" if isinstance(node.max_val, float) else str(node.max_val)
        print(f"{indent}Depth {node.depth}: [{min_str}, {max_str}] ({len(node.points)} points)")
        if node.left:
            self.print_tree(node.left, level + 1)
        if node.right:
            self.print_tree(node.right, level + 1)

In [7]:
#testing cluster
clus = DivisiveCluster(max_depth=16)
clus.fit(values_X[0])
print(clus.get_clusters()[:3])
clus.fit(values_X[1])
print(clus.get_clusters()[:3])

[['e', 'e', 'e', 'e', 'e', 'e', 'e', 'e', 'e', 'e', 'e', 'e', 'e', 'e', 'e', 'e', 'e', 'e', 'e', 'e', 'e', 'e', 'e', 'e', 'e', 'e', 'e', 'e', 'e', 'e', 'e', 'e', 'e', 'e', 'e', 'e', 'e', 'e', 'e', 'e', 'e', 'e', 'e', 'e', 'e', 'e', 'e', 'e', 'e', 'e', 'e', 'e', 'e', 'e', 'e', 'e', 'e', 'e', 'e', 'e', 'e', 'e', 'e', 'e', 'e', 'e', 'e', 'e', 'e', 'e', 'e', 'e', 'e', 'e', 'e', 'e', 'e', 'e', 'e', 'e', 'e', 'e', 'e', 'e', 'e', 'e', 'e', 'e', 'e', 'e', 'e', 'e', 'e', 'e', 'e', 'e', 'e', 'e', 'e', 'e', 'e', 'e', 'e', 'e', 'e', 'e', 'e', 'e', 'e', 'e', 'e', 'e', 'e', 'e', 'e', 'e', 'e', 'e', 'e', 'e', 'e', 'e', 'e', 'e', 'e', 'e', 'e', 'e', 'e', 'e', 'e', 'e', 'e', 'e', 'e', 'e', 'e', 'e', 'e', 'e', 'e', 'e', 'e', 'e', 'e', 'e', 'e', 'e', 'e', 'e', 'e', 'e', 'e', 'e', 'e', 'e', 'e', 'e', 'e', 'e', 'e', 'e', 'e', 'e', 'e', 'e', 'e', 'e', 'e', 'e', 'e', 'e', 'e', 'e', 'e', 'e', 'e', 'e', 'e', 'e', 'e', 'e', 'e', 'e', 'e', 'e', 'e', 'e', 'e', 'e', 'e', 'e', 'e', 'e', 'e', 'e', 'e', 'e', 'e', 'e'

In [71]:
#BINARY CONVERTION CODE

In [72]:
import numpy as np
import math
from typing import List, Any

# Assumes DivisiveCluster is already defined

def bin_convertion(n_array, max_bins=16) -> np.ndarray:
    """
    Convert a 1D array to one‑hot encoded strings.
    The string length equals the actual number of bins used, which is:
        - len(unique_values) if <= max_bins
        - number of clusters (≤ max_bins) otherwise
    Returns a 1D numpy array of strings.
    """
    array = np.array(n_array)
    unique_values = np.unique(array)

    if len(unique_values) <= max_bins:
        # Direct mapping without clustering
        value_to_idx = {val: i for i, val in enumerate(unique_values)}
        indices = np.array([value_to_idx[val] for val in array])
        n_bins = len(unique_values)
    else:
        # Cluster into at most max_bins groups
        depth = int(math.ceil(math.log2(max_bins)))
        cluster = DivisiveCluster(max_depth=depth, min_cluster_size=1)
        print(f"Clustering array (unique: {len(unique_values)}) with depth {depth}")
        cluster.fit(array)
        clusters = cluster.get_clusters()

        value_to_cluster = {}
        for cluster_idx, cluster_values in enumerate(clusters):
            for value in cluster_values:
                value_to_cluster[value] = cluster_idx

        indices = np.array([value_to_cluster[val] for val in array])
        n_bins = len(clusters)

    # Build one‑hot strings of length n_bins (not max_bins)
    onehot_strings = []
    for idx in indices:
        arr = ['0'] * n_bins
        arr[idx] = '1'
        onehot_strings.append(''.join(arr))

    return np.array(onehot_strings)


def bin_convertion_2d(array_2d, max_bins=16) -> np.ndarray:
    """
    Process each feature (row) and return a 2D array where each row is a sample
    and each column is the one‑hot string for the corresponding feature.
    String lengths vary per feature (actual number of bins used).
    Shape: (n_samples, n_features)
    """
    processed_features = []
    for feature_row in array_2d:
        onehot_strings = bin_convertion(feature_row, max_bins=max_bins)
        processed_features.append(onehot_strings)

    # Transpose so rows = samples, columns = features
    return np.array(processed_features).T

In [73]:
#TEST BIN CONVERSion
print(bin_convertion(values_X[0], max_bins=8))
converted = bin_convertion_2d(values_X, max_bins=14)

['10' '10' '01' ... '10' '01' '01']


In [74]:
def flatten_binary_strings(bin_strings):
    combined = ''.join(bin_strings)
    return [int(ch) for ch in combined]

In [75]:
flattened_samples = np.array([flatten_binary_strings(row) for row in converted])
print(flattened_samples)

[[1 0 0 ... 0 0 0]
 [1 0 1 ... 0 0 0]
 [0 1 0 ... 1 0 0]
 ...
 [1 0 0 ... 0 0 1]
 [0 1 0 ... 0 0 0]
 [0 1 0 ... 0 0 0]]


In [76]:
from pydl85 import DL85Cluster
import numpy as np

# Assuming X_flat is your binned feature matrix (n_samples, n_features_binary)
# X_flat = ... (your flattened binary data)

# Initialize the clusterer
clusterer = DL85Cluster(max_depth=3, min_sup=5, time_limit=60, verbose=True)

# Fit the model — notice there is NO 'y' parameter
clusterer.fit(flattened_samples)

# Predict cluster labels for each sample
cluster_labels = clusterer.predict(flattened_samples)

print(f"Assigned clusters: {cluster_labels}")

Assigned clusters: [0.24, 0.24, 0.24, 0.21, 0.24, 0.24, 0.24, 0.24, 0.24, 0.24, 0.24, 0.24, 0.24, 0.21, 0.24, 0.21, 0.24, 0.24, 0.24, 0.24, 0.24, 0.24, 0.24, 0.24, 0.24, 0.24, 0.24, 0.24, 0.25, 0.24, 0.24, 0.24, 0.24, 0.24, 0.25, 0.24, 0.24, 0.25, 0.24, 0.24, 0.24, 0.24, 0.24, 0.24, 0.24, 0.24, 0.24, 0.24, 0.24, 0.24, 0.24, 0.24, 0.24, 0.24, 0.24, 0.21, 0.24, 0.24, 0.24, 0.24, 0.24, 0.24, 0.24, 0.24, 0.21, 0.24, 0.24, 0.24, 0.24, 0.25, 0.24, 0.25, 0.24, 0.24, 0.25, 0.25, 0.24, 0.24, 0.24, 0.21, 0.24, 0.24, 0.21, 0.24, 0.21, 0.24, 0.24, 0.24, 0.24, 0.24, 0.24, 0.24, 0.24, 0.21, 0.24, 0.24, 0.24, 0.24, 0.24, 0.21, 0.24, 0.24, 0.24, 0.24, 0.24, 0.24, 0.24, 0.24, 0.24, 0.24, 0.24, 0.25, 0.24, 0.24, 0.24, 0.24, 0.24, 0.25, 0.24, 0.24, 0.24, 0.24, 0.21, 0.24, 0.21, 0.24, 0.21, 0.24, 0.24, 0.24, 0.24, 0.25, 0.24, 0.25, 0.24, 0.25, 0.24, 0.24, 0.24, 0.24, 0.24, 0.24, 0.24, 0.24, 0.21, 0.21, 0.24, 0.24, 0.24, 0.24, 0.21, 0.24, 0.24, 0.25, 0.24, 0.24, 0.24, 0.24, 0.24, 0.24, 0.24, 0.24, 0.24, 0.